# Transformer and MoE From Scratch

This notebook is the main assignment document for the project. It is written as a CS189-style implementation assignment: every section connects a mathematical object to tensor shapes, implementation instructions, common bugs, and tests.

The goal is to understand a decoder-only transformer well enough to implement the dense model by hand, then replace only the feed-forward network with a Mixture of Experts (MoE) module. Attention, normalization, residual connectors, logits, loss, checkpointing, generation, and the training loop should remain shared between dense and MoE models.

## Assignment Contract

There are two implementation tracks in this repository.

**Manual track.** Files under `src/moe_transformer_lab/manual/` are for learning the forward and backward passes. You should implement or study these components without relying on PyTorch autograd for the layer internals: `Linear`, `Embedding`, `SoftmaxCrossEntropy`, `LayerNorm`, causal attention, dense MLP, residual connector, transformer block, and the small decoder-only LM.

**Trainable track.** Files under `src/moe_transformer_lab/trainable/` are practical PyTorch `nn.Module` implementations used for local training. The dense and MoE models share the same transformer scaffold. MoE uses PyTorch autograd because the router, top-k selection, dispatch, and scatter/combine logic are already enough architecture practice for this assignment.

A correct solution should satisfy three ideas:

1. Every tensor shape is known before code is written.
2. The dense transformer can be tested one layer at a time.
3. MoE is a feed-forward swap, not a separate model family.

## Big Picture: The Full Pipeline

A language model is trained to answer one repeated question: given the tokens so far, what token should come next? Everything in this assignment exists to make that next-token prediction more accurate.

The complete training and generation paths use the same model forward pass in different ways.

![Training versus generation paths](../assets/figures/training_vs_generation_paths.svg)

**What to notice.** Training consumes shifted targets and updates weights. Generation has no targets and feeds each sampled token back into the next step.

**What Problem Is a Language Model Solving?** If the text is `the cat sat`, the model sees prefixes like `the`, then `the cat`, then `the cat sat`, and learns to predict the next token at every position. In code, this is why training batches use shifted sequences: `x = tokens[t : t + N]` and `y = tokens[t + 1 : t + N + 1]`.

**Where MoE Fits.** MoE changes only the FFN/MoE box inside each transformer block. It does not change tokenization, embeddings, attention masks, the language-model loss, optimizer, checkpoint format, or generation loop.

![Pipeline overview](../assets/figures/pipeline_overview.svg)

**What to notice.** The training path and generation path share the same model forward pass. Training compares logits to shifted targets; generation reads only the last-position logits and samples the next token.

Keep this pipeline in mind as you implement each piece. A layer is not an isolated formula; it is one stage in the flow from text to loss and from prompt to generated text.

## Project 01 Additive Track: Transformer Inference Foundations

This additive track keeps the original Transformer + MoE assignment intact and layers the summer Project 01 goals on top of it. The core question is no longer only can I implement a Transformer and swap in MoE. It is also can I explain and measure the model path used by LLM inference systems.

The Project 01 path emphasizes the dense decoder-only model first: token IDs, embeddings, causal attention, residual blocks, logits, generation, prefill/decode, and KV-cache memory. MoE remains an optional feed-forward extension after those foundations are clear.

**Success criteria.** By the end of this track, you should be able to explain every attention and KV-cache tensor shape without looking at the code, run a tiny overfit experiment, generate from a checkpoint, compare cached and uncached decode, and write a short benchmark report with latency, tokens/sec, memory, and limitations.

**Current implementation boundary.** Some Project 01 topics are lecture and roadmap items in this pass. RoPE, RMSNorm, SwiGLU, top-p sampling, KV cache, and benchmarking are planned implementation milestones, not hidden completed features.

### Mini-Lecture: Decoder-Only LM Tensor Shapes

A decoder-only language model receives integer token IDs and returns one vector of vocabulary scores for every position. The model is trained so position `t` predicts token `t+1`.

The basic flow is a shape ladder from integer token IDs to one vocabulary-score vector per position.

![Decoder-only shape ladder](../assets/figures/decoder_shape_ladder.svg)

**What to notice.** Transformer blocks preserve `(B, T, C)`. The LM head is where the final dimension changes from hidden width `C` to vocabulary size `V`.

| Symbol | Code Name | Shape | Meaning |
| --- | --- | --- | --- |
| `B` | `batch_size` | scalar | number of sequences processed together |
| `T` | `seq_len` or `block_size` | scalar | tokens per sequence |
| `C` | `d_model` | scalar | residual-stream width |
| `V` | `vocab_size` | scalar | number of possible next tokens |
| `idx` | `idx` | `(B, T)` | integer token IDs |
| `x` | hidden states | `(B, T, C)` | token representations |
| `logits` | logits | `(B, T, V)` | unnormalized next-token scores |
| `targets` | targets | `(B, T)` | shifted next-token IDs |

**Explanation.** The embedding table has shape `(V, C)`. Indexing it with `(B, T)` token IDs returns `(B, T, C)`. Transformer blocks preserve `(B, T, C)`. The LM head maps the last dimension from `C` to `V`, so logits are `(B, T, V)`. Cross entropy flattens logits to `(B*T, V)` and targets to `(B*T)`.

**Common bugs.**

- Treating `T` as the batch axis when flattening for loss.
- Computing softmax over the sequence axis instead of the vocabulary axis.
- Forgetting that generation only uses `logits[:, -1, :]`, while training uses every position.

**Check yourself.**

1. If `B=4`, `T=128`, `C=256`, and `V=257`, what is the logits shape?
2. Why does the target tensor not have a vocabulary dimension?
3. Which dimension changes in the LM head?

**Command.**

```powershell
python -m pytest tests/test_swappable_ffn.py::test_lm_forward_works_for_dense_and_moe
```

### Mini-Lecture: Causal Multi-Head Attention Shapes

Attention is the sequence-mixing operation. For each token position, a query asks which earlier token positions are useful. Keys are compared with the query, values are averaged according to the resulting probabilities, and the output returns to the residual stream.

Starting from hidden states `x` with shape `(B, T, C)`, a linear projection produces Q, K, and V. After splitting into heads, each tensor has shape `(B, H, T, Dh)`, where `Dh = C / H`.

| Quantity | Shape | Meaning |
| --- | --- | --- |
| `x` | `(B, T, C)` | residual-stream input |
| `q`, `k`, `v` before split | `(B, T, C)` | full-width projected tensors |
| `q`, `k`, `v` after split | `(B, H, T, Dh)` | per-head tensors |
| `scores` | `(B, H, T, T)` | query-key similarity for every target/source position pair |
| `mask` | `(T, T)` | lower-triangular allowed-attention pattern |
| `attn` | `(B, H, T, T)` | row-normalized attention probabilities |
| `head_out` | `(B, H, T, Dh)` | weighted values per head |
| `y` | `(B, T, C)` | merged attention output |

![Causal mask matrix](../assets/figures/causal_mask_matrix.svg)

**What to notice.** Green cells are allowed source positions. Red cells are future tokens and must be blocked before softmax.

**Explanation.** Every target position has a query, and every source position has a key. The score at row `i`, column `j` asks whether position `i` should read from position `j`. Causal decoding forbids `j > i`, because that would leak future target tokens.

**Common bugs.**

- Using an upper-triangular mask when the model needs a lower-triangular mask.
- Forgetting `.contiguous()` before reshaping after transposes.
- Applying softmax before the mask, which gives future positions probability mass.

**Check yourself.**

1. If `C=256` and `H=4`, what is `Dh`?
2. Why do scores have shape `(B, H, T, T)` rather than `(B, T, C)`?
3. Which tensor creates the `O(T^2)` memory pressure?

**Command.**

```powershell
python -m pytest tests/test_manual_attention.py
```

### Mini-Lecture: Why Attention Memory Is O(T^2)

The expensive part of standard attention is not the Q, K, or V tensor by itself. The quadratic term appears when every query position compares against every key position. For each batch item and head, the model forms a matrix with `T` rows and `T` columns.

For one layer, ignoring dtype details and smaller tensors:

```text
attention_scores_memory ~= B * H * T * T * dtype_bytes
attention_probs_memory  ~= B * H * T * T * dtype_bytes
```

![Attention memory scaling](../assets/figures/attention_memory_scaling.svg)

**What to notice.** Prefill compares every prompt position against every prompt position. Cached decode computes a much thinner score row for the newest token.

| Scenario | Query Length | Key/Value Length | Score Shape | Scaling |
| --- | ---: | ---: | --- | --- |
| Training or full prompt prefill | `T` | `T` | `(B, H, T, T)` | quadratic |
| One-token cached decode | `1` | `T_cache` | `(B, H, 1, T_cache)` | linear per token |
| Uncached generation step | `T_now` | `T_now` | `(B, H, T_now, T_now)` | repeats quadratic work |

**Explanation.** During training and prefill, the model attends over the full sequence at once. During decode with a KV cache, the newest query attends over cached keys and values. The newest score tensor is much smaller, but the cache still grows linearly with context length.

**Common bugs.**

- Saying KV cache makes attention constant-time. It removes repeated K/V computation, but the newest token still reads the cached sequence.
- Measuring only generated tokens/sec without recording prompt length.
- Comparing cached and uncached generation with different sampling settings.

**Check yourself.**

1. Why is prefill still expensive even if a decode cache will be used later?
2. What changes from `(T, T)` to `(1, T_cache)` during single-token decode?
3. Why does long-context serving care so much about attention kernels and KV memory?

**Planned command.**

```powershell
python scripts/benchmark_inference.py --config configs/tiny_inference.json
```

### Mini-Lecture: RoPE Intuition

Learned absolute position embeddings add a position vector to each token representation. Rotary position embeddings, usually called RoPE, instead rotate query and key features by an angle that depends on token position. The rotation happens inside attention, after Q and K are projected and split into heads, before computing `QK^T`.

The important implementation path is visualized below.

![RoPE Q/K rotation](../assets/figures/rope_qk_rotation.svg)

**What to notice.** Position enters the Q/K dot product. V is not rotated because values are the content being mixed.

| Quantity | Shape | RoPE Role |
| --- | --- | --- |
| `q` | `(B, H, T, Dh)` | rotated by position |
| `k` | `(B, H, T, Dh)` | rotated by position |
| `v` | `(B, H, T, Dh)` | not rotated |
| `cos`, `sin` cache | `(T, Dh/2)` or broadcastable variant | reused rotation factors |

**Explanation.** The dot product between a rotated query at position `i` and a rotated key at position `j` depends on the difference between their rotation angles. Since those angles are determined by positions, attention receives relative-distance information without adding a separate learned position vector to the residual stream.

**Common bugs.**

- Rotating V. RoPE applies to Q and K, not V.
- Building a cache with the wrong sequence length for decode positions.
- Forgetting that `Dh` must be even for pairwise rotation.
- Applying RoPE before splitting heads and then reshaping incorrectly.

**Check yourself.**

1. Why are Q and K rotated but V is not?
2. Where should RoPE sit relative to the Q/K linear projections?
3. During cached decode, why does the newest token need the correct absolute position index?

**Planned test.**

```powershell
python -m pytest tests/test_rope.py
```

### Mini-Lecture: RMSNorm And SwiGLU

The current trainable model uses LayerNorm and a GELU MLP, which are good educational defaults. Many modern decoder-only LLMs use RMSNorm and SwiGLU-style feed-forward blocks. These are Project 01 modernization targets.

**RMSNorm.** LayerNorm subtracts the mean and divides by the standard deviation. RMSNorm does not subtract the mean; it scales by the root mean square of the features. It keeps a learned scale parameter but usually has no bias.

```text
rms = sqrt(mean(x_i^2) + eps)
y = weight * x / rms
```

![RMSNorm versus LayerNorm](../assets/figures/rmsnorm_vs_layernorm.svg)

**What to notice.** Both normalization choices preserve `(B, T, C)`. RMSNorm skips the mean-subtraction branch.

**SwiGLU.** A standard MLP expands then contracts: `Linear -> GELU -> Linear`. SwiGLU uses two input projections. One produces values, the other produces a gate. The gated hidden state is then projected back to the model width.

```text
value = W_v x
gate = silu(W_g x)
y = W_o (value * gate)
```

![SwiGLU MLP](../assets/figures/swiglu_mlp.svg)

**What to notice.** The gate controls which expanded features pass through before the output projection returns to the residual width.

| Component | Current Repo | Project 01 Modern Variant | Shape Contract |
| --- | --- | --- | --- |
| Norm | LayerNorm | RMSNorm | `(B, T, C) -> (B, T, C)` |
| MLP | GELU MLP | SwiGLU MLP | `(B, T, C) -> (B, T, C)` |
| Residual block | pre-norm | pre-norm | preserves residual stream |

**Common bugs.**

- Treating RMSNorm as LayerNorm without mean subtraction.
- Forgetting that SwiGLU needs two input projections, not one.
- Changing the block interface when the output shape should remain `(B, T, C)`.

**Check yourself.**

1. What statistic does RMSNorm use that LayerNorm also uses?
2. What statistic does RMSNorm skip?
3. Why can SwiGLU replace the current dense MLP without changing the residual path?

**Planned tests.**

```powershell
python -m pytest tests/test_modern_block.py
```

### Mini-Lecture: Language Modeling Objective And Shifted Targets

A decoder-only language model learns next-token prediction. For a token sequence `[t0, t1, t2, t3]`, the input might be `[t0, t1, t2]` and the targets are `[t1, t2, t3]`. The model predicts the next token at every position in parallel during training.

| Tensor | Example | Shape |
| --- | --- | --- |
| `idx` | `[t0, t1, t2]` | `(B, T)` |
| `targets` | `[t1, t2, t3]` | `(B, T)` |
| `logits` | scores for each next token | `(B, T, V)` |
| flattened logits | training view | `(B*T, V)` |
| flattened targets | training labels | `(B*T)` |

![Training step flow](../assets/figures/training_step_flow.svg)

**What to notice.** Dense and MoE models share the same next-token loss. MoE adds an auxiliary router term, but the shifted-target objective remains unchanged.

**Explanation.** Cross entropy compares each row of logits against the correct target token ID. It averages over the valid `(batch, position)` pairs. If MoE is enabled, the total training loss is cross entropy plus the router auxiliary loss.

**Common bugs.**

- Training the model to reconstruct the same token instead of the next token.
- Shifting inputs and targets inconsistently between training and evaluation.
- Reporting only total MoE loss without separately logging cross entropy and auxiliary loss.

**Check yourself.**

1. Why can training predict all next tokens in parallel, while generation must loop?
2. What axis does cross entropy treat as classes?
3. Why does MoE not change the language-model objective?

**Command.**

```powershell
python -m pytest tests/test_tiny_overfit.py
```

### Mini-Lecture: Sampling Controls

Generation turns the final-position logits into one next token, appends it, and repeats. The sampling policy strongly changes output behavior even when model weights are fixed.

| Method | Rule | Typical Failure Mode |
| --- | --- | --- |
| Greedy | choose the largest logit | repetitive or bland text |
| Temperature | divide logits by temperature before softmax | too low becomes deterministic; too high becomes noisy |
| Top-k | keep only the `k` highest logits | can exclude good tail tokens if `k` is too small |
| Top-p | keep the smallest set whose probability mass exceeds `p` | unstable if combined carelessly with tiny or flat distributions |

![Sampling filters](../assets/figures/sampling_filters.svg)

**What to notice.** Temperature changes sharpness, while top-k and top-p remove candidate vocabulary tokens before sampling.

**Explanation.** Greedy decoding uses `argmax` and is deterministic. Temperature changes the sharpness of the distribution. Top-k filters to a fixed number of vocabulary candidates. Top-p sorts by probability and keeps the smallest prefix whose cumulative probability reaches `p`, so the number of available tokens adapts to model confidence.

**Common bugs.**

- Applying top-p to logits before softmax without accounting for probability mass.
- Letting all tokens be filtered out.
- Comparing generation outputs without setting the random seed.
- Treating top-k sampling and MoE router top-k as the same concept.

**Check yourself.**

1. Which sampling method is deterministic by default?
2. Why does top-p sometimes keep many tokens and sometimes only a few?
3. What is the difference between vocabulary top-k and MoE router top-k?

**Command.**

```powershell
python scripts/generate.py --checkpoint runs/tinystories_moe/ckpt_last.pt --prompt "Once upon a time" --temperature 0.9 --top-k 50
```

### Mini-Lecture: Prefill Vs Decode

LLM inference has two different phases. Prefill processes the whole prompt. Decode generates one new token at a time. They stress the system differently.

| Phase | Input Tokens This Step | Main Work | Output Needed |
| --- | ---: | --- | --- |
| Prefill | full prompt length `T_prompt` | run all layers over the prompt | logits for the final prompt position and KV cache |
| Decode | usually `1` new token | run all layers for the newest token using cached K/V | logits for the next token and updated cache |

![Prefill decode timeline](../assets/figures/prefill_decode_timeline.svg)

**What to notice.** Prefill happens once for the prompt. Decode repeats once per generated token and grows the cache each step.

**Explanation.** The prompt tokens are known, so prefill can process them in one forward pass. Standard attention builds `(T_prompt, T_prompt)` score matrices per head. Decode then loops once per generated token. Without a cache, the model recomputes keys and values for the entire growing sequence at every step. With a cache, each layer stores previous K/V tensors and computes only the newest token's Q/K/V.

**Common bugs.**

- Benchmarking decode without separating prompt prefill time.
- Thinking cached decode avoids attending to previous tokens entirely. It reuses K/V, but the newest query still reads them.
- Forgetting to update positions during decode. This matters for RoPE and positional embeddings.

**Check yourself.**

1. Which phase sees the full prompt at once?
2. Which phase repeats once per generated token?
3. Why should a benchmark report time to first token or prefill latency separately from decode tokens/sec?

**Planned command.**

```powershell
python scripts/benchmark_inference.py --prompt-length 128 --max-new-tokens 32
```

### Mini-Lecture: KV-Cache Shape Contract And Memory Formula

A KV cache stores keys and values from previous tokens for each transformer layer. During decode, the model appends the newest K and V to the cache and reuses the old tensors instead of recomputing them.

A common cache layout is one `(key, value)` pair per layer:

| Cache Item | Shape | Meaning |
| --- | --- | --- |
| `past_key[layer]` | `(B, H, T_cache, Dh)` | cached keys for previous tokens |
| `past_value[layer]` | `(B, H, T_cache, Dh)` | cached values for previous tokens |
| `new_key` | `(B, H, 1, Dh)` | key for newest token |
| `new_value` | `(B, H, 1, Dh)` | value for newest token |
| `present_key` | `(B, H, T_cache + 1, Dh)` | updated key cache |
| `present_value` | `(B, H, T_cache + 1, Dh)` | updated value cache |

![KV cache layout](../assets/figures/kv_cache_layout.svg)

**What to notice.** The cache is per layer and stores both K and V, so the memory formula has a factor of `layers * 2`.

The approximate cache memory is:

```text
bytes = layers * 2 * B * H * T_cache * Dh * dtype_bytes
```

The factor `2` is for K and V. Since `H * Dh = C`, the same formula can be written:

```text
bytes = layers * 2 * B * T_cache * C * dtype_bytes
```

**Concrete example.** With `layers=4`, `B=1`, `H=4`, `T_cache=128`, `Dh=32`, and fp16 cache entries, memory is `4 * 2 * 1 * 4 * 128 * 32 * 2 = 262144` bytes, or about `0.25 MiB`. Large models and long contexts scale this quickly.

**Common bugs.**

- Dropping the layer dimension from the memory formula.
- Forgetting the factor of `2` for keys and values.
- Concatenating cache tensors on the wrong sequence axis.
- Comparing cached and uncached logits while dropout is enabled.

**Check yourself.**

1. What is stored per layer?
2. Why does cache memory grow linearly with sequence length?
3. Why must cached-vs-uncached correctness tests run in eval mode?

**Planned test.**

```powershell
python -m pytest tests/test_kv_cache.py
```

### Mini-Lecture: Cached Vs Uncached Generation Correctness

A KV cache is a performance optimization. It should not change the model's next-token logits except for tiny floating-point differences. The most important correctness test compares the final-token logits from two paths.

| Path | Work Done | Expected Result |
| --- | --- | --- |
| Full forward | recomputes all prompt K/V | reference final-token logits |
| Cached prefill/decode | reuses previous K/V | logits close to reference |

![Cached versus uncached decode](../assets/figures/cached_vs_uncached_decode.svg)

**What to notice.** The cached path is allowed to be faster, not different. Compare logits in eval mode before trusting sampled text.

**Explanation.** The uncached path runs the full prompt through the model and reads `logits[:, -1, :]`. The cached path runs prefill to build cache, then runs decode for the newest token using `past_key_values`. The test should use the same weights, same token IDs, same dtype when possible, `model.eval()`, and deterministic settings.

**Common bugs.**

- Comparing logits at the wrong position.
- Passing the whole prompt again during decode instead of only the newest token.
- Incorrect position index for the decode token.
- Updating only K but not V, or mixing layer order in the cache list.

**Check yourself.**

1. Why compare logits instead of sampled text?
2. Which token's logits should the uncached path use?
3. Why can a sampling seed hide a cache correctness bug?

**Planned test.**

```powershell
python -m pytest tests/test_kv_cache.py::test_cached_decode_matches_uncached_final_token
```

### Mini-Lecture: Benchmark Vocabulary

Inference benchmarks should separate correctness, speed, memory, and workload shape. A single tokens/sec number is not enough.

| Metric | Meaning | Why It Matters |
| --- | --- | --- |
| Prefill latency | time to process the prompt | controls time to first token for long prompts |
| Decode latency | time per generated token or per decode step | controls streaming responsiveness |
| Tokens/sec | generated-token throughput | useful for capacity and cost estimates |
| Peak memory | maximum allocated memory | determines feasible batch/context/model size |
| Speedup | cached vs uncached ratio | shows whether cache work paid off |
| Prompt length | number of input tokens | changes prefill cost and cache size |
| New tokens | number of generated tokens | changes decode-loop duration |

![Benchmark metrics dashboard](../assets/figures/benchmark_metrics_dashboard.svg)

**What to notice.** A benchmark row is only interpretable if it records workload shape, cache mode, hardware, and memory alongside speed.

**Explanation.** Record hardware, device, dtype, model config, prompt length, generated tokens, random seed, and exact command. Warm up the model before timing if using GPU. Keep raw JSON/CSV logs in `results/` and summarize in `reports/transformer_inference_report.md`.

**Common bugs.**

- Timing tokenizer work together with model execution without saying so.
- Reporting GPU memory without synchronizing CUDA.
- Comparing runs with different prompt lengths or different max-new-token counts.
- Treating a tiny CPU result as evidence about production serving.

**Check yourself.**

1. Which metric best describes time to first token?
2. Why should prompt length appear in every result table?
3. What does cached-vs-uncached speedup fail to prove about production serving?

**Planned command.**

```powershell
python scripts/benchmark_inference.py --config configs/tiny_inference.json --out results/tiny_inference.jsonl
```

## 0. Setup

**What You Do.** Install the project, import Torch, and confirm that tests can run.

```powershell
python -m pip install -e .[dev,train]
```

If PyTorch is not installed yet, install it from the official selector: https://pytorch.org/get-started/locally/

**Common Bugs.**

- Installing PyTorch into a different Python environment than the one running Jupyter.
- Using Python 3.13 on Windows before PyTorch wheels are available for your CUDA setup. Python 3.12 is the safer default.
- Running notebook cells from a directory other than the repository root.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT / 'src'))

import torch
torch.manual_seed(0)
print(torch.__version__)

## 1. Shape Discipline and Gradient Checks

**Mathematical Object.** A language-model batch contains integer token ids and predicts the next token at each position.

| Symbol | Code Name | Shape | Meaning |
| --- | --- | --- | --- |
| `B` | `batch_size` | scalar | number of sequences |
| `N` | `seq_len` or `block_size` | scalar | tokens per sequence |
| `D` | `d_model` | scalar | embedding dimension |
| `V` | `vocab_size` | scalar | vocabulary size |
| `idx` | `idx` | `(B, N)` | token ids |
| `X` | `x` | `(B, N, D)` | token representations |
| `L` | `logits` | `(B, N, V)` | unnormalized token scores |

**What You Implement.** Before writing any layer, write down the input shape, output shape, parameter shape, and gradient shape. The manual tests use finite differences:

`dJ/dtheta_i ~= (J(theta_i + eps) - J(theta_i - eps)) / (2 eps)`

**Common Bugs.**

- Forgetting that a token-wise layer acts on the last dimension and preserves `(B, N)`.
- Summing gradients over the feature dimension when you meant to sum over batch and sequence.
- Comparing float32 finite differences with too small an epsilon. Use float64 in gradient-check tests.

**Run This Test.**

```powershell
python -m pytest tests/test_manual_layers.py::test_linear_backward_matches_finite_difference
```

In [ ]:
!python -m pytest tests/test_manual_layers.py::test_linear_backward_matches_finite_difference

## 2. Linear Layer

**Before You Implement.** A linear layer is the basic learned coordinate change used throughout the transformer. It turns token representations into new feature spaces: embeddings are projected into queries, keys, and values; attention outputs are projected back to `D`; MLPs expand from `D` to `D_ff` and contract back to `D`; the LM head projects from hidden states to vocabulary logits.

**Where This Appears In The Pipeline.** Linear layers appear inside attention, inside the dense MLP, inside every expert MLP, and at the final `lm_head` stage before cross entropy.

![Matrix multiplication](../assets/figures/matrix_multiplication.svg)

**What to notice.** One output entry is one dot product: a row of `X` with a column of `W`. The same rule applies whether `X` represents examples, tokens, or flattened `(B*N)` token positions.

**Mathematical Object.** Implement the affine map

`Y = XW + b`

For a token-wise transformer layer, `X` may have shape `(B, N, D_in)`. The layer should behave as if the first two axes were flattened to `M = B * N` examples.

| Quantity | Shape |
| --- | --- |
| `X` | `(B, N, D_in)` or `(M, D_in)` |
| `W` | `(D_in, D_out)` |
| `b` | `(D_out,)` |
| `Y` | `(B, N, D_out)` or `(M, D_out)` |
| `dW` | `(D_in, D_out)` |
| `db` | `(D_out,)` |
| `dX` | same as `X` |

**What You Implement.** In `src/moe_transformer_lab/manual/layers.py::Linear`, implement `forward` and `backward`. Cache the input `x` during forward. During backward, flatten all leading dimensions, accumulate `dW = X^T dY`, accumulate `db = sum_rows dY`, and return `dX = dY W^T` reshaped to the input shape.

![Linear backward](../assets/figures/linear_backward.svg)

**What to notice.** Backward uses the upstream gradient `dY` three ways: with `X` to get `dW`, by summing to get `db`, and with `W^T` to return `dX` to the previous layer.

**Questions.**

1. How many learnable parameters does the layer have with bias?
2. Why does the backward pass sum over batch and sequence for `db`?
3. Why is `dX` needed if the linear layer is not the first layer?

**Common Bugs.**

- Using `W @ X` instead of `X @ W`.
- Forgetting to accumulate into `.grad` when a parameter is reused.
- Returning flattened `dX` instead of the original input shape.

## 3. Embeddings and Cross Entropy

**Before You Implement.** Raw text is not numerical. The tokenizer first maps text to token ids, then the embedding table maps each id to a vector the transformer can process. Cross entropy appears at the other end of the model: after the LM head produces one score per vocabulary item, the loss rewards the score assigned to the true next token.

**Where This Appears In The Pipeline.** Embeddings are the first learned stage after tokenization. Cross entropy is the final supervised training signal before the backward pass.

![Shifted targets](../assets/figures/embedding_shifted_targets.svg)

**What to notice.** The input and target sequences contain almost the same tokens, but the target is shifted one position. Position `n` predicts token `n+1`.

**Mathematical Object.** A token embedding table is `E in R^{V x D}`. Token id `i` selects row `E_i`. The output projection maps final hidden states to vocabulary logits.

| Quantity | Shape | Meaning |
| --- | --- | --- |
| `idx` | `(B, N)` | token ids |
| `E` | `(V, D)` | embedding table |
| `X` | `(B, N, D)` | selected embeddings |
| `logits` | `(B, N, V)` | token scores |
| `targets` | `(B, N)` | next-token ids |

**What You Implement.**

- `Embedding.forward`: return `weight[indices]`.
- `Embedding.backward`: use row accumulation because the same token may appear many times.
- `SoftmaxCrossEntropy.forward`: subtract the row maximum before exponentiating.
- `SoftmaxCrossEntropy.backward`: return `(p - one_hot(target)) / number_of_valid_tokens`.

**Common Bugs.**

- Overwriting an embedding gradient row instead of adding to it.
- Computing softmax over the sequence axis instead of the vocabulary axis.
- Forgetting that the loss is averaged, so the gradient must also be divided by the number of targets.

**Run This Test.**

```powershell
python -m pytest tests/test_manual_layers.py::test_softmax_cross_entropy_backward_rows_sum_to_zero
```

In [ ]:
!python -m pytest tests/test_manual_layers.py::test_softmax_cross_entropy_backward_rows_sum_to_zero

## 4. Layer Normalization

**Mathematical Object.** Layer normalization normalizes each token vector independently over its feature dimension:

`mu = mean_i x_i`, `sigma^2 = mean_i (x_i - mu)^2`, `xhat = (x - mu) / sqrt(sigma^2 + eps)`, `y = gamma * xhat + beta`.

| Quantity | Shape |
| --- | --- |
| `x` | `(B, N, D)` |
| `gamma` | `(D,)` |
| `beta` | `(D,)` |
| `y` | `(B, N, D)` |
| `dgamma`, `dbeta` | `(D,)` |

**What You Implement.** In `LayerNorm`, cache `xhat` and `inv_std`. The backward pass should sum parameter gradients over every axis except the last one. Compare your result to PyTorch autograd.

**Common Bugs.**

- Normalizing over batch or sequence instead of features.
- Forgetting `eps` in the forward pass.
- Computing `dgamma` with shape `(B, N, D)` instead of reducing to `(D,)`.

**Run This Test.**

```powershell
python -m pytest tests/test_manual_layers.py::test_layer_norm_backward_matches_autograd
```

In [ ]:
!python -m pytest tests/test_manual_layers.py::test_layer_norm_backward_matches_autograd

## 5. Causal Mask and Attention

**Before You Implement.** Attention is the operation that lets a token representation depend on earlier tokens. This is what makes the meaning of a word context-dependent. In `I went to the bank`, attention gives the representation of `bank` a way to look back at `went`, `to`, and other context tokens.

**Why Attention?** A token-wise MLP alone cannot mix information across positions. Attention computes a soft lookup table: each query asks which keys are relevant, the row-softmax converts scores into weights, and the weighted sum of values returns context.

**Where This Appears In The Pipeline.** Attention is the sequence-mixing half of every transformer block. The causal mask is what preserves the next-token prediction rule by preventing a token from seeing future answers.

![Attention computation](../assets/figures/attention_qk_softmax_v.svg)

**What to notice.** The mask is applied before softmax. After row-softmax, the attention matrix `A` contains nonnegative weights that sum to one across each row.

![Multi-head attention](../assets/figures/multihead_attention.svg)

**What to notice.** Multi-head attention repeats the same attention operation in several smaller subspaces, then concatenates the head outputs and projects back to `D`.

**Mathematical Object.** Attention is a soft lookup table. For one head:

`Z = Q K^T / sqrt(D_k)`, `A = SoftMaxRow(Z)`, `Y = A V`.

In a decoder-only language model, row `n` may attend only to columns `i <= n`. A causal mask sets future scores to `-inf` before softmax.

| Quantity | Single Sequence Shape | Batched Multi-Head Shape |
| --- | --- | --- |
| `Q` | `(N, D_k)` | `(B, H, N, D_h)` |
| `K` | `(N, D_k)` | `(B, H, N, D_h)` |
| `V` | `(N, D_v)` | `(B, H, N, D_h)` |
| `Z` | `(N, N)` | `(B, H, N, N)` |
| `A` | `(N, N)` | `(B, H, N, N)` |
| `Y` | `(N, D_v)` | `(B, H, N, D_h)` |

**What You Implement.**

- `causal_mask(N)`: lower-triangular boolean matrix.
- `masked_row_softmax`: row-wise softmax where masked entries receive probability zero.
- `ScaledDotProductAttention`: cache `Q`, `K`, `V`, and `A`; implement backward through matrix multiply and softmax.
- `MultiHeadSelfAttention`: project `X` into `Q`, `K`, `V`, split heads, apply attention, merge heads, and apply the output projection.

**Questions.**

1. Why does each row of `A` sum to 1?
2. Why divide by `sqrt(D_k)`?
3. Why is the mask applied before softmax rather than after?
4. What is the dominant computational cost as `N` grows?

**Common Bugs.**

- Applying softmax over the wrong axis.
- Using an upper-triangular mask instead of lower-triangular.
- Forgetting to scale by `sqrt(D_k)`.
- Merging heads with a non-contiguous view after transposing.

**Run This Test.**

```powershell
python -m pytest tests/test_manual_attention.py
```

In [ ]:
!python -m pytest tests/test_manual_attention.py

## 6. Dense MLP and Residual Connector

**Before You Implement.** A transformer block has two jobs. Attention mixes information across token positions. The MLP then applies a learned nonlinear transformation to each token independently. Without the MLP, attention would mostly form weighted averages of value vectors; the block would have less capacity to build new features at each position.

**Why Residual Connections and Normalization?** Residual connections give gradients a short path through deep networks, and layer normalization keeps token representations on a stable scale. Pre-norm transformers usually train more reliably because each sublayer receives normalized input while the residual stream can carry information forward.

**Where This Appears In The Pipeline.** This is the repeated middle stage: embeddings enter the first block, then each block updates the residual stream, and the final hidden states go to the LM head.

![Transformer block](../assets/figures/transformer_block.svg)

**What to notice.** The residual stream bypasses each sublayer and is added back afterward. Dense FFN and MoE occupy the same box in the second branch.

**Mathematical Object.** A transformer block combines sequence mixing and token-wise nonlinear processing. Attention mixes information across tokens. The MLP processes each token independently:

`MLP(x) = W_2 GELU(W_1 x + b_1) + b_2`.

Use the pre-norm residual connector:

```python
x = x + attention(norm1(x), mask)
x = x + mlp(norm2(x))
```

| Component | Input Shape | Output Shape |
| --- | --- | --- |
| `norm1` | `(B, N, D)` | `(B, N, D)` |
| `attention` | `(B, N, D)` | `(B, N, D)` |
| `norm2` | `(B, N, D)` | `(B, N, D)` |
| `mlp` | `(B, N, D)` | `(B, N, D)` |

**What You Implement.**

- `ManualMLP`: `Linear -> GELU -> Linear`.
- `ManualTransformerBlock`: two pre-norm residual branches.
- `ManualDecoderOnlyTransformer`: token embeddings, positional embeddings, repeated blocks, final norm, LM head, and cross-entropy.

**Common Bugs.**

- Applying layer norm after the residual when the implementation expects pre-norm.
- Dropping one residual path in backward.
- Forgetting position embeddings, which makes attention permutation-equivariant and unable to represent order.

## 7. Swappable Feed-Forward Interface

**Key Design Rule.** The dense MLP and MoE block must have the same interface. The transformer block should not know whether it is using dense or MoE feed-forward computation.

```python
y, aux_loss = feed_forward(x, train=True)
```

`DenseMLP` returns `aux_loss = 0`. `MoEFeedForward` returns a router load-balancing loss. The block stays fixed:

```python
x = x + attention(norm1(x))
ffn_out, aux_loss = feed_forward(norm2(x))
x = x + ffn_out
```

**Why This Matters.** MoE is not changing the language-model objective. It is not changing attention. It is not changing generation. It only changes how the token-wise feed-forward sublayer computes its `(B, N, D)` output.

**What You Implement.** In `src/moe_transformer_lab/trainable/model.py`, keep `TransformerBlock.forward` shared. If you add new feed-forward variants later, they should follow the same `(output, aux_loss)` contract.

**Run This Test.**

```powershell
python -m pytest tests/test_swappable_ffn.py
```

In [ ]:
!python -m pytest tests/test_swappable_ffn.py

## 8. MoE Intuition

**Before You Implement.** In many transformers, the feed-forward network contains a large fraction of the model parameters. MoE asks whether every token needs to use the same large FFN. Instead, the model stores several expert FFNs and activates only a small subset for each token.

**Why MoE?** MoE separates total parameters from active parameters. More experts increase the model's storage capacity, while top-k routing keeps per-token compute closer to the cost of one or two FFNs. This is useful when you want more specialization without multiplying the compute for every token by the number of experts.

**Where This Appears In The Pipeline.** MoE replaces the token-wise FFN subpipeline inside a transformer block. The residual stream still enters with shape `(B, N, D)` and leaves with shape `(B, N, D)`.

![MoE routing](../assets/figures/moe_routing.svg)

**What to notice.** The router produces expert probabilities for every token, but only the top-k experts are executed for that token.

A dense transformer MLP sends every token through the same large feed-forward network. A Mixture of Experts layer has several expert MLPs and a router. For each token, the router chooses a small number of experts. Only those experts process that token.

This gives a model more total parameters without increasing the active computation per token by the same factor. If there are `E = 8` experts and `k = 1`, each token uses one expert, but the model can store eight different feed-forward transformations.

The hard engineering part is not the expert MLP. Each expert is ordinary `DenseMLP`. The hard part is routing tokens to experts and then putting the outputs back in the original token order.

## 9. MoE Tensor Flow

The MoE subpipeline is:

```text
x -> flatten tokens -> router -> top-k experts
  -> expert MLPs -> weighted combine -> reshape -> residual path
```

![MoE dispatch and combine](../assets/figures/moe_dispatch_combine.svg)

**What to notice.** Flattening makes routing easier, but the final output must be reshaped back to `(B, N, D)` so it can rejoin the residual path.

Let `E` be the number of experts, `k` be the number of selected experts per token, and `T = B * N` be the number of tokens in the flattened batch.

| Quantity | Shape | Meaning |
| --- | --- | --- |
| `x` | `(B, N, D)` | block input after `norm2` |
| `flat_x` | `(T, D)` | tokens flattened across batch and sequence |
| `router_logits` | `(T, E)` | unnormalized expert scores |
| `router_probs` | `(T, E)` | softmax probabilities over experts |
| `top_probs` | `(T, k)` | selected expert probabilities |
| `top_indices` | `(T, k)` | selected expert ids |
| `flat_y` | `(T, D)` | combined expert outputs in flattened order |
| `y` | `(B, N, D)` | output reshaped back to token layout |
| `aux_loss` | scalar | load-balancing penalty |

The output must have exactly the same shape as the dense MLP output. That is the whole reason MoE can be swapped into the existing transformer block.

## 10. MoE Implementation Steps

**File.** `src/moe_transformer_lab/trainable/model.py`

**Class.** `MoEFeedForward`

**Method.** `forward(self, x, *, train=True)`

Implement the forward pass in this exact order:

1. Save `original_shape = x.shape`.
2. Flatten tokens: `flat_x = x.reshape(-1, D)`.
3. Compute router scores: `router_logits = router(flat_x)`.
4. Convert scores to expert probabilities: `router_probs = softmax(router_logits, dim=-1)`.
5. Select experts: `top_probs, top_indices = topk(router_probs, k=top_k, dim=-1)`.
6. Renormalize `top_probs` so selected probabilities sum to 1 for each token.
7. Allocate `flat_y = zeros_like(flat_x)`.
8. For each expert id, find all `(token_position, top_slot)` pairs where `top_indices == expert_id`.
9. Gather those tokens: `expert_in = flat_x[token_positions]`.
10. Run the expert MLP: `expert_out, _ = expert(expert_in)`.
11. Weight expert outputs by `top_probs[token_positions, top_slots]`.
12. Add weighted outputs back with `flat_y.index_add_(0, token_positions, weighted_output)`.
13. Compute `aux_loss` from `router_probs` and `top_indices`.
14. Return `flat_y.reshape(original_shape), aux_loss`.

**Why `index_add_`?** If `top_k > 1`, the same token can receive output from multiple experts. Scatter assignment would overwrite one contribution. `index_add_` sums all expert contributions for each token.

## 11. MoE Pseudocode

This pseudocode matches the current reference implementation. Read it line by line and check the shape after each line.

```python
def forward(x):
    original_shape = x.shape          # (B, N, D)
    flat_x = x.reshape(-1, D)         # (T, D)

    router_logits = router(flat_x)    # (T, E)
    router_probs = softmax(router_logits, dim=-1)

    top_probs, top_indices = topk(router_probs, k=top_k, dim=-1)
    top_probs = top_probs / top_probs.sum(dim=-1, keepdim=True)

    flat_y = zeros_like(flat_x)       # (T, D)

    for expert_id, expert in enumerate(experts):
        token_positions, top_slots = where(top_indices == expert_id)
        if token_positions.numel() == 0:
            continue

        expert_in = flat_x[token_positions]
        expert_out, _ = expert(expert_in)
        weights = top_probs[token_positions, top_slots].unsqueeze(-1)
        flat_y.index_add_(0, token_positions, expert_out * weights)

    aux_loss = load_balance_loss(router_probs, top_indices)
    return flat_y.reshape(original_shape), aux_loss
```

**Check Yourself.** If `B = 2`, `N = 5`, `D = 16`, `E = 4`, and `k = 1`, then `T = 10`, `router_logits` has shape `(10, 4)`, and `top_indices` has shape `(10, 1)`.

## 12. MoE Load-Balancing Loss

Without a balancing penalty, the router can collapse and send most tokens to one expert. Then the other experts receive little training signal.

The reference implementation uses a Switch-style auxiliary loss. Let:

- `selected_fraction[e]` be the fraction of top-k slots assigned to expert `e`.
- `prob_fraction[e]` be the average router probability for expert `e`.

The auxiliary loss is:

`aux_loss = aux_loss_coef * E * sum_e selected_fraction[e] * prob_fraction[e]`

**Interpretation.** This loss is small when router probabilities and actual selected experts are spread more evenly. It does not force exact uniform routing, but it discourages one-expert collapse.

**Common Bugs.**

- Computing the loss from only `top_probs`, which ignores unselected router probabilities.
- Forgetting to multiply by `aux_loss_coef`, causing the auxiliary loss to dominate cross entropy.
- Treating `top_indices` as differentiable. Expert selection is discrete; gradients flow through router probabilities used in the loss and selected top-k weights.

## 13. Why MoE Uses Autograd Here

Dense layers, normalization, softmax, and attention are smooth functions, so handwritten backward passes are a good learning exercise. Top-k routing is different: the identity of the selected experts changes discontinuously when router probabilities cross.

In this assignment, you should understand the gradient story at a high level:

- Expert MLP parameters receive gradients from the tokens routed to them.
- Router probabilities receive gradients through selected expert weights and through the auxiliary loss.
- The discrete top-k indices are treated as routing decisions, not smooth variables.

That is why the manual-backward track stops at the dense transformer fundamentals, while the trainable MoE path uses PyTorch autograd.

## 14. MoE Debugging Checklist

Use this checklist before blaming the optimizer.

**Shape mismatches.** Print `x.shape`, `flat_x.shape`, `router_logits.shape`, `top_indices.shape`, and `flat_y.shape`. The final output must equal the original `(B, N, D)` shape.

**Wrong softmax axis.** Router softmax must be over experts, so use `dim=-1` on `(T, E)`.

**Unnormalized top-k weights.** After top-k, selected probabilities should sum to 1 for each token. Check `top_probs.sum(dim=-1)`.

**Scatter overwrite.** Use `index_add_`, not assignment, so top-2 routing can sum multiple expert outputs for the same token.

**Router collapse.** Count selected experts with `torch.bincount(top_indices.reshape(-1), minlength=n_experts)`. If one expert dominates, increase `aux_loss_coef` slightly or train longer.

**Aux loss scale.** Print `ce_loss` and `aux_loss` separately. The auxiliary loss should regularize routing, not swamp cross entropy.

**Run These Tests.**

```powershell
python -m pytest tests/test_swappable_ffn.py
python -m pytest tests/test_tiny_overfit.py
```

## 15. One Training Step, End to End

A training step turns a sampled chunk of text into a parameter update.

![Training step flow](../assets/figures/training_step_flow.svg)

**What to notice.** The optimizer never sees text directly. It sees gradients produced from next-token cross entropy and, for MoE, the auxiliary router loss.

**Forward.** The batch `x` has token ids with shape `(B, N)`. The model converts ids to embeddings, adds position embeddings, runs transformer blocks, and produces `logits` with shape `(B, N, V)`.

**Loss.** The target batch `y` is the same text shifted by one token. Cross entropy compares `logits[b, n, :]` against `y[b, n]`. For MoE models, the final training loss is `cross_entropy + aux_loss`.

**Backward.** In the manual track, you explicitly call the backward methods for educational layers. In the trainable track, PyTorch records the computation graph and `loss.backward()` fills `.grad` tensors.

**Optimizer Step.** Gradient clipping limits unusually large updates. AdamW then updates parameters. `optimizer.zero_grad(set_to_none=True)` clears old gradients so the next step starts cleanly.

**Validation and Checkpointing.** Validation runs the same forward/loss computation without parameter updates. A checkpoint stores model weights, config, optimizer state, and step number so training or generation can resume later.

## 16. One Generation Step, End to End

Generation uses the trained model autoregressively. There are no targets and no backward pass.

![Generation loop](../assets/figures/generation_loop.svg)

**What to notice.** The loop consumes only the final-position logits, samples one token, appends it, and repeats with a longer context.

**Context Cropping.** The model was trained with a maximum context length. If the prompt grows longer than `max_seq_len`, generation keeps only the most recent tokens.

**Last-Position Logits.** The model outputs logits for every position, but generation only needs the final position because that position predicts the next token after the current prompt.

**Temperature.** Dividing logits by a lower temperature makes the distribution sharper and more deterministic. A higher temperature makes sampling more random.

**Top-k Filtering.** Keeping only the top `k` logits removes very low-probability tokens before softmax. This often improves small-model samples by avoiding extremely unlikely choices.

**Sampling.** The next token is sampled from the probability distribution, appended to the sequence, and decoded back to text at the end.

## 17. Local TinyStories Training

**First Run Path.** Start with the dependency-free byte tokenizer. It is not the most efficient tokenizer, but it keeps the pipeline simple.

```powershell
python scripts/download_tinystories.py --out-dir data/tinystories/raw
python scripts/train_tokenizer.py --kind byte --raw-dir data/tinystories/raw --out data/tinystories/tokenizer.json
```

**Tiny smoke test.** Use this to verify data loading, model construction, forward/backward, checkpointing, and validation logging.

```powershell
python scripts/train_dense.py --steps 20 --eval-interval 10 --block-size 64 --batch-size 4 --d-model 128 --n-layers 2 --n-heads 4 --d-ff 512 --max-train-chars 200000 --max-valid-chars 50000
```

**12GB dense baseline.**

```powershell
python scripts/train_dense.py --steps 1000 --eval-interval 100 --block-size 128 --batch-size 16 --grad-accum 2 --d-model 256 --n-layers 4 --n-heads 4 --d-ff 1024 --amp
```

**12GB MoE run.** This changes only the feed-forward type and expert settings.

```powershell
python scripts/train_moe.py --ffn moe --steps 1000 --eval-interval 100 --block-size 128 --batch-size 16 --grad-accum 2 --d-model 256 --n-layers 4 --n-heads 4 --d-ff 1024 --n-experts 4 --top-k 1 --aux-loss-coef 0.01 --amp
```

**Expected Output.** Training prints lines like `step 100 train_loss=... valid_loss=...` and saves `runs/tinystories_moe/ckpt_last.pt`. Loss should generally decrease on the tiny smoke test. Longer TinyStories runs may still produce rough text because the model is intentionally small.

**Generate.**

```powershell
python scripts/generate.py --checkpoint runs/tinystories_moe/ckpt_last.pt --prompt "Once upon a time" --max-new-tokens 120
```

**When to Scale Up.**

- Increase `block-size` from 128 to 256 when memory allows and validation loss is still improving.
- Increase `d-model` or `n-layers` for better quality, but expect slower training.
- Increase `n-experts` after top-1 MoE works; do not start with many experts before the router behavior is debugged.
- Use `grad-accum` to keep the effective batch size larger when VRAM is tight.

## 18. Acceptance Criteria

You are done when the following checks pass in the environment where PyTorch is installed:

```powershell
python -m pytest tests/test_manual_layers.py
python -m pytest tests/test_manual_attention.py
python -m pytest tests/test_swappable_ffn.py
python -m pytest tests/test_tiny_overfit.py
python -m pytest tests/test_checkpoint.py
```

Conceptually, you should also be able to answer these questions:

1. Which tensors change shape when attention splits into heads?
2. Why does the dense MLP preserve `(B, N, D)`?
3. Why can MoE replace the dense MLP without changing the transformer block interface?
4. What does `index_add_` do in the MoE combine step?
5. Why is the router auxiliary loss needed?
6. Which parts of the assignment use handwritten backward, and which parts use PyTorch autograd?

In [ ]:
!python -m pytest tests/test_checkpoint.py